# 03.01 Why Graphics Needs Cross Product

在 Chapter 02 中，我们已经学习了 dot product。

dot product 的核心作用是衡量两个方向之间的 alignment：

$$
u \cdot v = ||u|| ||v|| \cos(\theta)
$$

如果 u 和 v 都是单位向量，那么 dot product 直接等于两个方向夹角的 cos 值。

因此 dot product 很适合回答这类问题：

- 两个方向是否同向？
- 两个方向是否垂直？
- 表面是否朝向光源？
- 目标是否在相机前方？
- 点在某个方向上的投影长度是多少？

但是 dot product 不能回答另一个图形学中极其重要的问题：

```text
两个方向共同张成的平面，应该朝哪一侧？

## 03.01.01 Dot Product Is Not Enough

假设我们有两个三维向量 u 和 v。

dot product 给出的是一个 scalar：

$$
u \cdot v
$$

这个 scalar 告诉我们 u 和 v 的夹角关系。

但是它不会告诉我们：

- u 和 v 所在平面的法线方向；
- 三角形的正面朝向哪边；
- 顶点顺序是 clockwise 还是 counter-clockwise；
- 相机坐标系中的 right / up / forward 如何互相构造；
- 一个旋转方向是顺时针还是逆时针。

换句话说：

```text
dot product answers: how aligned are two directions?
cross product answers: what direction is perpendicular to both directions?

## 03.01.02 The Core Graphics Problem: Getting a Surface Normal

在图形学中，一个三角形通常由三个点表示：

$$
p_0,\ p_1,\ p_2
$$

为了计算这个三角形的表面法线，我们先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

这两条边都在三角形所在的平面上。

现在问题变成：

```text
如何找到一个同时垂直于 e1 和 e2 的方向？

## 03.01.03 A Concrete Numerical Example

考虑一个位于 xy 平面上的三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

构造两条边：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

根据右手坐标系约定：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以这个三角形的 normal 指向正 z 方向。

如果交换顺序：

$$
e_2 \times e_1=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

方向会完全反过来。

这说明 cross product 不满足交换律：

$$
u \times v=-(v \times u)
$$

## 03.01.04 Why Direction Matters in Graphics

cross product 的方向不是一个纯数学细节，而是直接影响图形学结果。

如果三角形 normal 方向错了，会影响：

1. lighting  
   表面可能被错误地认为背对光源。

2. back-face culling  
   本来应该显示的三角形可能被剔除。

3. shading  
   法线方向错误会导致明暗反转。

4. collision detection  
   接触面方向错误会导致响应方向错误。

5. camera basis  
   right、up、forward 构造顺序错误会导致相机坐标系翻转。

因此 cross product 的方向约定必须和坐标系约定一起理解。

## 03.01.05 Cross Product as Orientation Information

dot product 的结果是 scalar。

cross product 的结果是 vector。

这两者的几何意义不同：

```text
dot product:
two vectors → scalar
measures alignment

cross product:
two 3D vectors → vector
produces a perpendicular direction

## 03.01.06 Graphics Applications Preview

后续章节中，cross product 会反复出现。

第一类应用是 surface normal。

给定三角形三个点：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

$$
n=\operatorname{normalize}(e_1 \times e_2)
$$

第二类应用是 camera basis。

如果我们知道 camera forward 和 world up，可以构造 camera right：

$$
r=\operatorname{normalize}(f \times up)
$$

然后再构造真正的 camera up：

$$
u=\operatorname{normalize}(r \times f)
$$

第三类应用是 back-face culling。

如果 triangle normal 和 view direction 的关系表明三角形背对相机，那么可以不绘制它。

第四类应用是面积。

cross product 的长度和两个向量张成的平行四边形面积有关：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

三角形面积则是它的一半：

$$
A=\frac{1}{2}||e_1 \times e_2||
$$

## 03.01.07 NumPy Example

下面先用 NumPy 验证最基本的 cross product 方向。

我们使用：

```text
x × y = z
y × x = -z

In [ ]:
import numpy as np

x = np.array([1.0, 0.0, 0.0])
y = np.array([0.0, 1.0, 0.0])
z = np.array([0.0, 0.0, 1.0])

cross_xy = np.cross(x, y)
cross_yx = np.cross(y, x)

print("x cross y:", cross_xy)
print("y cross x:", cross_yx)
print("Expected z:", z)
print("Expected -z:", -z)

x cross y: [0. 0. 1.]
y cross x: [ 0.  0. -1.]
Expected z: [0. 0. 1.]
Expected -z: [-0. -0. -1.]


## 03.01.08 NumPy Example: Triangle Normal

现在考虑三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

先构造边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

再计算：

$$
n=e_1 \times e_2
$$

In [2]:
def normalize(v):
    """归一化向量；zero vector 不能归一化。"""
    v = np.asarray(v, dtype=float)
    length = np.linalg.norm(v)

    if np.isclose(length, 0.0):
        raise ValueError("Cannot normalize a zero vector.")

    return v / length


p0 = np.array([0.0, 0.0, 0.0])
p1 = np.array([1.0, 0.0, 0.0])
p2 = np.array([0.0, 1.0, 0.0])

e1 = p1 - p0
e2 = p2 - p0

n = np.cross(e1, e2)
n_hat = normalize(n)

print("e1:", e1)
print("e2:", e2)
print("normal:", n)
print("unit normal:", n_hat)

e1: [1. 0. 0.]
e2: [0. 1. 0.]
normal: [0. 0. 1.]
unit normal: [0. 0. 1.]


## 03.01.09 What Happens If We Reverse Vertex Order?

如果交换 p1 和 p2，相当于改变三角形的 winding order。

原来是：

$$
n=(p_1-p_0) \times (p_2-p_0)
$$

交换后是：

$$
n'=(p_2-p_0) \times (p_1-p_0)
$$

因为 cross product 反交换：

$$
n'=-n
$$

所以 normal direction 会翻转。

这就是 winding order 会影响 triangle normal 的原因。

## 03.01.11 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请判断：

$$
u \times v
$$

的方向。

再判断：

$$
v \times u
$$

的方向。

---

### Exercise 2

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

并判断：

$$
e_1 \times e_2
$$

的方向。

---

### Exercise 3

如果一个三角形的顶点顺序从：

$$
p0 → p1 → p2
$$

改成：
$$
p0 → p2 → p1
$$
请说明 normal direction 会发生什么变化。

# 03.02 Cross Product Definition

上一节我们已经知道，cross product 用来从两个 3D 向量构造一个新的方向。

这个新方向有两个关键性质：

1. 它垂直于第一个输入向量；
2. 它垂直于第二个输入向量。

也就是说，如果输入是 u 和 v，那么 cross product 输出的是：

$$
u \times v
$$

这个结果不是 scalar，而是一个 vector。

这和 dot product 完全不同：

```text
dot product:
u · v → scalar

cross product:
u × v → vector

## 03.02.01 Cross Product Only Makes Sense Here as a 3D Operation

在本课程的图形学语境中，我们主要讨论 3D cross product。

给定两个三维向量：

$$
u=
\begin{bmatrix}
u_x \\
u_y \\
u_z
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
v_x \\
v_y \\
v_z
\end{bmatrix}
$$

它们的 cross product 仍然是一个三维向量：

$$
u \times v=
\begin{bmatrix}
? \\
? \\
?
\end{bmatrix}
$$

这个输出向量的方向由右手定则决定。

它的长度和 u、v 张成的面积有关。

这一节先重点掌握代数公式。

## 03.02.02 Algebraic Definition

给定：

$$
u=
\begin{bmatrix}
u_x \\
u_y \\
u_z
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
v_x \\
v_y \\
v_z
\end{bmatrix}
$$

cross product 定义为：

$$
u \times v=
\begin{bmatrix}
u_y v_z-u_z v_y \\
u_z v_x-u_x v_z \\
u_x v_y-u_y v_x
\end{bmatrix}=
\begin{bmatrix}
D_{yz} \\
D_{zx} \\
D_{xy}
\end{bmatrix}
$$

也就是说：

$$
(u \times v)_x=u_y v_z-u_z v_y
$$

$$
(u \times v)_y=u_z v_x-u_x v_z
$$

$$
(u \times v)_z=u_x v_y-u_y v_x
$$

注意第二个分量很容易写错。

它是：

$$
u_z v_x-u_x v_z
$$

不是：

$$
u_x v_z-u_z v_x
$$

## 03.02.03 Concrete Example: x Cross y

令：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

根据公式：

$$
x \times y=
\begin{bmatrix}
0 \cdot 0-0 \cdot 1 \\
0 \cdot 0-1 \cdot 0 \\
1 \cdot 1-0 \cdot 0
\end{bmatrix}
$$

所以：

$$
x \times y=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

这正好是 z 方向。

因此在右手坐标系中：

$$
x \times y=z
$$

## 03.02.04 Reversing the Order

现在交换顺序：

$$
y \times x
$$

其中：

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

根据公式：

$$
y \times x=
\begin{bmatrix}
1 \cdot 0-0 \cdot 0 \\
0 \cdot 1-0 \cdot 0 \\
0 \cdot 0-1 \cdot 1
\end{bmatrix}
$$

所以：

$$
y \times x=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

也就是说：

$$
y \times x=-z
$$

因此：

$$
u \times v=-(v \times u)
$$

cross product 不满足交换律。

## 03.02.05 Cross Product Is Orthogonal to Both Inputs

cross product 的核心性质是：

$$
u \times v \perp u
$$

同时：

$$
u \times v \perp v
$$

换成 dot product 表示就是：

$$
(u \times v) \cdot u=0
$$

$$
(u \times v) \cdot v=0
$$

这就是为什么 cross product 可以用来计算 surface normal。

如果三角形的两条边是：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

那么：

$$
n=e_1 \times e_2
$$

会同时垂直于 e1 和 e2。

因此 n 垂直于三角形所在平面。

## 03.02.06 Numerical Example with Non-Axis Vectors

令：

$$
u=
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
4 \\
5 \\
6
\end{bmatrix}
$$

根据公式：

$$
u \times v=
\begin{bmatrix}
2 \cdot 6-3 \cdot 5 \\
3 \cdot 4-1 \cdot 6 \\
1 \cdot 5-2 \cdot 4
\end{bmatrix}
$$

所以：

$$
u \times v=
\begin{bmatrix}
-3 \\
6 \\
-3
\end{bmatrix}
$$

验证它和 u 垂直：

$$
(u \times v) \cdot u=(-3)\cdot 1+6\cdot 2+(-3)\cdot 3
$$

$$
(u \times v) \cdot u=0
$$

验证它和 v 垂直：

$$
(u \times v) \cdot v=(-3)\cdot 4+6\cdot 5+(-3)\cdot 6
$$

$$
(u \times v) \cdot v=0
$$

## 03.02.07 NumPy Implementation: Using np.cross

NumPy 已经提供了 cross product 函数：

```text
np.cross(u, v)

In [3]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

cross_uv = np.cross(u, v)
cross_vu = np.cross(v, u)

print("u cross v:", cross_uv)
print("v cross u:", cross_vu)
print("-(v cross u):", -cross_vu)

print("Are they opposite?", np.allclose(cross_uv, -cross_vu))

u cross v: [-3.  6. -3.]
v cross u: [ 3. -6.  3.]
-(v cross u): [-3.  6. -3.]
Are they opposite? True


## 03.02.10 Graphics Connection: Triangle Normal

cross product 最直接的图形学应用是计算三角形法线。

给定三角形三个点：

$$
p_0,\ p_1,\ p_2
$$

先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

然后：

$$
n=e_1 \times e_2
$$

最后归一化：

$$
\hat{n}=\frac{n}{||n||}
$$

这里必须注意：

```text
e1 × e2 和 e2 × e1 的方向相反。

## 03.02.11 Degenerate Triangle Case

如果三角形的三个点共线，那么两条边不能张成一个真正的平面。

例如：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

此时：

$$
e_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

e1 和 e2 共线。

所以：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

zero vector 不能 normalized。这类三角形称为 degenerate triangle，在图形学中，它没有有效的 surface normal。

## 03.02.14 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请计算：

$$
u \times v
$$

并判断结果指向哪个坐标轴方向。

---

### Exercise 2

给定：

$$
u=
\begin{bmatrix}
1 \\
2 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
3 \\
1 \\
0
\end{bmatrix}
$$

请手动计算：

$$
u \times v
$$

并说明为什么结果只在 z 方向上有分量。

---

### Exercise 3

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

判断这个 normal 指向正 z 方向还是负 z 方向。

---

### Exercise 4

为什么下面三个点不能定义出一个有效的三角形 normal？

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
2 \\
2
\end{bmatrix}
$$

请从 cross product 的角度解释。

---

### Exercise 5

用 NumPy 实现一个函数：

```text
triangle_normal(p0, p1, p2)

# 03.03 Right-Hand Rule and Direction

cross product 最容易出错的地方不是数值计算，而是方向。

对于 dot product：

$$
u \cdot v=v \cdot u
$$

交换顺序不会改变结果。

但对于 cross product：

$$
u \times v=-(v \times u)
$$

交换顺序会让方向完全反过来。

因此，在图形学中使用 cross product 时，必须同时关注两件事：

1. 输入向量的顺序；
2. 当前使用的坐标系约定。

本项目采用 right-handed coordinate system。

在这个约定下：

$$
x \times y=z
$$

## 03.03.01 The Right-Hand Rule

右手定则用于判断 cross product 的方向。

对于：

$$
u \times v
$$

可以这样理解：

```text
右手四指从 u 的方向弯向 v 的方向，
大拇指指向的方向就是 u × v 的方向。

## 03.03.02 Standard Axes in a Right-Handed Coordinate System

在 right-handed coordinate system 中，标准基向量满足：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

并且：

$$
x \times y=z
$$

$$
y \times z=x
$$

$$
z \times x=y
$$

这三个关系可以看作右手坐标系中的正向循环。

反过来：

$$
y \times x=-z
$$

$$
z \times y=-x
$$

$$
x \times z=-y
$$

所以 cross product 的顺序不能随便交换。

## 03.03.03 Concrete Example: x Cross z

令：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

根据 right-handed coordinate system 的循环关系：

$$
z \times x=y
$$

因此反过来：

$$
x \times z=-y
$$

所以：

$$
x \times z=
\begin{bmatrix}
0 \\
-1 \\
0
\end{bmatrix}
$$

这个例子很重要，因为很多人会直觉上误以为 x × z 等于 y。

但在右手坐标系下，正确结果是：

$$
x \times z=-y
$$

## 03.03.04 Numerical Verification with the Formula

用公式验证：

$$
u \times v=
\begin{bmatrix}
u_y v_z-u_z v_y \\
u_z v_x-u_x v_z \\
u_x v_y-u_y v_x
\end{bmatrix}
$$

令：

$$
u=x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

则：

$$
x \times z=
\begin{bmatrix}
0 \cdot 1-0 \cdot 0 \\
0 \cdot 0-1 \cdot 1 \\
1 \cdot 0-0 \cdot 0
\end{bmatrix}
$$

所以：

$$
x \times z=
\begin{bmatrix}
0 \\
-1 \\
0
\end{bmatrix}
$$

这和右手定则一致。

## 03.03.05 Orientation of a Triangle

cross product 的方向会决定 triangle normal 的方向。

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

构造两条边：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

因此：

$$
n=e_1 \times e_2=x \times y=z
$$

所以 normal 指向正 z 方向。

如果交换 p1 和 p2：

$$
e_1'=p_2-p_0
$$

$$
e_2'=p_1-p_0
$$

则：

$$
n'=e_1' \times e_2'=y \times x=-z
$$

normal 会翻转到负 z 方向。

## 03.03.06 Winding Order and Normal Direction

三角形顶点顺序通常称为 winding order。

常见的 winding order 包括：

```text
counter-clockwise winding
clockwise winding

## 03.03.09 Graphics Application: Camera Basis Direction

cross product 也常用于构造 camera basis。

假设我们有：

- camera forward direction；
- approximate world up direction。

在右手坐标系中，如果 camera forward 指向屏幕里面，也就是类似 OpenGL 中的负 z 方向：

$$
f=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

world up 为：

$$
up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

那么可以用：

$$
r=\operatorname{normalize}(f \times up)
$$

得到 right direction。

计算结果是：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

然后再用：

$$
u=\operatorname{normalize}(r \times f)
$$

得到真正的 camera up direction。

注意：camera basis 的 cross product 顺序和你采用的 forward 方向约定有关。后续讲 View Matrix 时会专门处理这个问题。

## 03.03.11 Why This Matters for Lighting

在 Lambert lighting 中，我们使用：

$$
I=\max(0,\ n \cdot l)
$$

其中 n 是 surface normal，l 是从 surface point 指向 light source 的单位方向。

如果 triangle normal 因为顶点顺序错误而反向，那么原本应该被照亮的面可能变暗。

例如：

$$
n=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

光照方向为：

$$
l=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

那么：

$$
n \cdot l=1
$$

表面完全朝向光源。

如果 normal 翻转：

$$
n'=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

则：

$$
n' \cdot l=-1
$$

Lambert intensity 会变成：

$$
I=\max(0,\ -1)=0
$$

这说明 normal direction 错误会直接造成 lighting 错误。

## 03.03.13 Exercises

### Exercise 1

在 right-handed coordinate system 中，判断下面结果：

$$
x \times y
$$

$$
y \times z
$$

$$
z \times x
$$

分别指向哪个坐标轴方向。

---

### Exercise 2

在 right-handed coordinate system 中，判断下面结果：

$$
y \times x
$$

$$
z \times y
$$

$$
x \times z
$$

分别指向哪个坐标轴方向。

---

### Exercise 3

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

判断 normal 指向正 y 方向还是负 y 方向。

---

### Exercise 4

如果一个三角形的 normal 指向错误，可能会对 lighting 和 back-face culling 造成什么影响？

请分别用一句话说明。

---

### Exercise 5

假设：

$$
f=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

$$
up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请计算：

$$
f \times up
$$

并判断它是否指向正 x 方向。

---

### Exercise 6

用 NumPy 验证：

$$
u \times v=-(v \times u)
$$

任选两个非零且不共线的 3D 向量即可。

# 03.04 Cross Product Magnitude and Area

上一节我们已经学习：

$$
u \times v
$$

的方向由 right-hand rule 决定。

但 cross product 不只包含方向信息，也包含大小信息。

它的长度满足：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

其中 theta 是 u 和 v 之间的夹角。

这个公式说明：

```text
cross product 的长度 = 两个向量张成的平行四边形面积

## 03.04.01 From Direction to Magnitude

cross product 的结果是一个 vector。

这个 vector 有两个方面：

1. direction  
   由 right-hand rule 决定。

2. magnitude  
   由两个输入向量张成的面积决定。

也就是说：

$$
u \times v
$$

不仅告诉我们“垂直方向朝哪边”，还告诉我们“这两个向量张开的面积有多大”。

如果 u 和 v 几乎同向，那么它们张开的面积很小。

如果 u 和 v 垂直，那么它们张开的面积最大。

如果 u 和 v 完全共线，那么它们张不开一个平面，面积为 0。

## 03.04.02 Cross Product Magnitude Formula

cross product 的长度公式是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

这里：

- ||u|| 是 u 的长度；
- ||v|| 是 v 的长度；
- theta 是 u 和 v 的夹角；
- sin(theta) 控制两个向量张开的程度。

这个公式和 dot product 的角度公式很像。

dot product 是：

$$
u \cdot v=||u|| ||v|| \cos(\theta)
$$

cross product magnitude 是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

所以可以这样理解：

```text
dot product uses cos(theta): measures alignment
cross product magnitude uses sin(theta): measures area / perpendicular spread

## 03.04.02 Cross Product Magnitude Formula

cross product 的长度公式是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

这里：

- ||u|| 是 u 的长度；
- ||v|| 是 v 的长度；
- theta 是 u 和 v 的夹角；
- sin(theta) 控制两个向量张开的程度。

这个公式和 dot product 的角度公式很像。

dot product 是：

$$
u \cdot v=||u|| ||v|| \cos(\theta)
$$

cross product magnitude 是：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

所以可以这样理解：

```text
dot product uses cos(theta): measures alignment
cross product magnitude uses sin(theta): measures area / perpendicular spread

## 03.04.04 Triangle Area

三角形面积是对应平行四边形面积的一半。

如果三角形由三个点 p0, p1, p2 定义，先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

这两条边张成一个平行四边形。

平行四边形面积是：

$$
||e_1 \times e_2||
$$

因此三角形面积是：

$$
A=\frac{1}{2}||e_1 \times e_2||
$$

这就是 cross product 在 mesh processing、collision detection、geometry analysis 中经常出现的原因。

## 03.04.05 Concrete Example: Perpendicular Vectors

令：

$$
u=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

这两个向量互相垂直，所以：

$$
\theta=90^\circ
$$

因此：

$$
\sin(\theta)=1
$$

cross product 为：

$$
u \times v=
\begin{bmatrix}
0 \\
0 \\
6
\end{bmatrix}
$$

所以：

$$
||u \times v||=6
$$

从面积角度看，u 和 v 张成的平行四边形是一个 2 × 3 的矩形。

因此面积也是：

$$
2 \times 3=6
$$

如果把它看作三角形的两条边，那么三角形面积是：

$$
A=\frac{1}{2}\times 6=3
$$

## 03.04.06 Concrete Example: Non-Perpendicular Vectors

令：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
1 \\
1 \\
0
\end{bmatrix}
$$

这两个向量的夹角是 45°。

先计算 cross product：

$$
u \times v=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以：

$$
||u \times v||=1
$$

再用长度公式验证。

u 的长度是：

$$
||u||=1
$$

v 的长度是：

$$
||v||=\sqrt{2}
$$

夹角为：

$$
\theta=45^\circ
$$

所以：

$$
||u|| ||v|| \sin(\theta)=1 \cdot \sqrt{2} \cdot \frac{\sqrt{2}}{2}
$$

得到：

$$
||u|| ||v|| \sin(\theta)=1
$$

这和 cross product 的长度一致。

## 03.04.08 Proof Sketch

cross product 的长度公式可以通过下面恒等式理解：

$$
||u \times v||^2=||u||^2||v||^2-(u \cdot v)^2
$$

又因为：

$$
u \cdot v=||u|| ||v|| \cos(\theta)
$$

所以：

$$
(u \cdot v)^2=||u||^2||v||^2\cos^2(\theta)
$$

代入得到：

$$
||u \times v||^2=||u||^2||v||^2-||u||^2||v||^2\cos^2(\theta)
$$

整理得到：

$$
||u \times v||^2=||u||^2||v||^2(1-\cos^2(\theta))
$$

因为：

$$
1-\cos^2(\theta)=\sin^2(\theta)
$$

所以：

$$
||u \times v||^2=||u||^2||v||^2\sin^2(\theta)
$$

两边开平方，得到：

$$
||u \times v||=||u|| ||v|| |\sin(\theta)|
$$

当 theta 取 0 到 pi 之间的夹角时，sin(theta) 非负，因此可以写成：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

## 03.04.13 Graphics Application: Mesh Area and Degenerate Triangles

cross product magnitude 在图形学中有很多实际用途。

第一，计算 triangle area。

对于 mesh 中的每个三角形：

$$
A_i=\frac{1}{2}||e_{1,i} \times e_{2,i}||
$$

整个 mesh 的表面积可以近似为所有三角形面积之和：

$$
A_{\text{mesh}}=\sum_i A_i
$$

第二，检测 degenerate triangles。

如果：

$$
||e_1 \times e_2|| \approx 0
$$

说明三角形面积接近 0。

这通常意味着：

- 三个顶点共线；
- 两个顶点重合；
- 三角形极度扁平；
- normal 无法稳定计算；
- shading 或 collision 可能出现异常。

第三，normal 计算前的有效性检查。

在计算：

$$
\hat{n}=\frac{e_1 \times e_2}{||e_1 \times e_2||}
$$

之前，必须检查：

$$
||e_1 \times e_2|| \ne 0
$$

否则就是 normalize zero vector。

## 03.04.14 Graphics Application: Area-Weighted Normals

在真实 mesh 中，一个顶点通常被多个三角形共享。

为了计算 vertex normal，可以把相邻三角形的 face normals 加权平均。

一种常见方法是 area-weighted normal。

如果某个 face 的未归一化法线是：

$$
n_i=e_{1,i} \times e_{2,i}
$$

它的长度正比于三角形面积：

$$
||n_i||=2A_i
$$

因此，直接累加未归一化的 face normal：

$$
n_{\text{vertex}}=\sum_i n_i
$$

相当于让面积更大的三角形对 vertex normal 贡献更大。

最后再 normalize：

$$
\hat{n}_{\text{vertex}}=\operatorname{normalize}(n_{\text{vertex}})
$$

这个思想在 smooth shading 中非常重要。

现在不需要完整实现，后续讲 lighting and shading 时会正式展开。

# 03.05 Cross Product and Orthogonality

cross product 的核心性质是：

$$
u \times v
$$

同时垂直于 u 和 v。

也就是说：

$$
(u \times v) \cdot u=0
$$

$$
(u \times v) \cdot v=0
$$

这也是 cross product 能用于计算 surface normal 的根本原因。

在图形学中，如果三角形的两条边是：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

那么：

$$
n=e_1 \times e_2
$$

会同时垂直于 e1 和 e2。

因此 n 垂直于三角形所在的平面。

## 03.05.01 What Orthogonality Means

两个非零向量 orthogonal，意思是它们互相垂直。

用 dot product 表示：

$$
u \cdot v=0
$$

注意：

```text
orthogonal 是几何关系。
dot product 是代数检测工具。

## 03.05.02 Cross Product Produces a Normal Direction

在三维空间中，如果两个非零向量 u 和 v 不共线，那么它们会张成一个平面。

cross product：

$$
u \times v
$$

给出一个垂直于这个平面的方向。

这个方向就是 normal direction。

因此：

```text
two non-parallel vectors define a plane
cross product gives a vector perpendicular to that plane

## 03.05.07 Orthogonality and Triangle Normals

给定三角形：

$$
p_0,\ p_1,\ p_2
$$

构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

三角形法线为：

$$
n=e_1 \times e_2
$$

因为 cross product 的正交性质：

$$
n \cdot e_1=0
$$

$$
n \cdot e_2=0
$$

所以 n 垂直于三角形的两条边。

只要 e1 和 e2 不共线，它们就张成一个平面，因此 n 垂直于整个三角形平面。

## 03.05.11 Graphics Application: Surface Normal for Lighting

在 Lambert lighting 中，核心公式是：

$$
I=\max(0,\ \hat{n} \cdot \hat{l})
$$

这里：

- n 是 surface normal；
- l 是从 surface point 指向 light source 的方向；
- 二者通常都需要 normalize。

cross product 用来从几何中构造 n。

dot product 用来计算 n 和 l 的 alignment。

所以在 lighting 中，cross product 和 dot product 分工明确：

```text
cross product:
construct surface normal from geometry

dot product:
measure how much the normal faces the light

## 03.05.12 Graphics Application: Camera Basis

cross product 的正交性质也用于构造 camera basis。

一个相机坐标系通常需要三条互相垂直的轴：

```text
right
up
forward
```

如果已知 forward direction 和 approximate up direction，可以用 cross product 构造 right direction。

## 03.05.16 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
2 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
2 \\
-1 \\
1
\end{bmatrix}
$$

请计算：

$$
w=u \times v
$$

然后验证：

$$
w \cdot u=0
$$

以及：

$$
w \cdot v=0
$$

---

### Exercise 2

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
1
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

并验证：

$$
n \cdot e_1=0
$$

$$
n \cdot e_2=0
$$

---

### Exercise 3

为什么 zero vector 不能作为 surface normal？

请从“方向”和“normalize”两个角度解释。

---

### Exercise 4

给定：

$$
u=
\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
2 \\
2 \\
2
\end{bmatrix}
$$

请判断：

$$
u \times v
$$

是否可以作为有效 normal。

---

### Exercise 5

用 NumPy 写一段代码，验证：

$$
u \times v
$$

同时垂直于 u 和 v。

要求：

- 使用 `np.cross()`；
- 使用 `@` 计算 dot product；
- 使用 `np.isclose()` 判断是否接近 0。

---

### Exercise 6

在 camera basis 中，为什么不能直接把 world up 当作 camera up？

请从 world up 可能不垂直于 forward direction 的角度解释。

---

### Exercise 7

假设你已经构造出三个相机方向：

```text
right
up
forward

# 03.06 Triangle Normals

在三维图形学中，三角形是最基本的 surface primitive。一个 mesh 通常由大量三角形组成，而每个三角形都需要一个 normal direction 来描述它的朝向。

triangle normal 直接影响：

- lighting；
- shading；
- back-face culling；
- collision response；
- surface orientation；
- mesh validation；
- vertex normal construction。

给定三角形三个点：

$$
p_0,\ p_1,\ p_2
$$

我们可以先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

然后用 cross product 得到未归一化法线：

$$
n_{\text{raw}}=e_1 \times e_2
$$

如果这个三角形不是 degenerate triangle，那么单位法线为：

$$
\hat{n}=\frac{n_{\text{raw}}}{||n_{\text{raw}}||}
$$

这就是 triangle normal 的基本计算方法。

## 03.06.01 Graphics Motivation

在 Chapter 02 中，我们学习了 Lambert lighting：

$$
I=\max(0,\ \hat{n} \cdot \hat{l})
$$

这里的 n 就是 surface normal。

## 03.06.02 Face Normal vs Vertex Normal

在图形学中，经常会区分两类 normal：

```text
face normal
vertex normal
```

face normal:
one normal per triangle

vertex normal 是顶点上的法线。一个顶点通常被多个三角形共享，因此 vertex normal 通常由相邻 face normals 平均得到。也就是 one normal per vertex

## 03.06.03 Mathematical Definition

给定三角形顶点：

$$
p_0=
\begin{bmatrix}
x_0 \\
y_0 \\
z_0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
x_1 \\
y_1 \\
z_1
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
x_2 \\
y_2 \\
z_2
\end{bmatrix}
$$

先定义两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

未归一化 triangle normal 定义为：

$$
n_{\text{raw}}=e_1 \times e_2
$$

如果：

$$
||n_{\text{raw}}|| \ne 0
$$

则单位 triangle normal 为：

$$
\hat{n}=\frac{n_{\text{raw}}}{||n_{\text{raw}}||}
$$

如果：

$$
||n_{\text{raw}}||=0
$$

说明三角形面积为 0，无法定义有效 normal。

## 03.06.04 Why the Normal Is Perpendicular to the Triangle

因为：

$$
n_{\text{raw}}=e_1 \times e_2
$$

根据 cross product 的正交性质：

$$
n_{\text{raw}} \cdot e_1=0
$$

$$
n_{\text{raw}} \cdot e_2=0
$$

如果 e1 和 e2 不共线，它们就张成三角形所在的平面。

一个向量如果同时垂直于这个平面上的两条不共线方向，那么它就垂直于整个平面。

因此，n_raw 是三角形所在平面的 normal direction。

归一化之后：

$$
\hat{n}=\frac{n_{\text{raw}}}{||n_{\text{raw}}||}
$$

仍然垂直于三角形平面，只是长度变成 1。

## 03.06.06 Vertex Order Changes Normal Direction

如果三角形顶点顺序是：

```text
p0 → p1 → p2

## 03.06.08 Degenerate Triangle

如果三角形三个点共线，或者有两个点重合，那么三角形面积为 0。

这时：

$$
n_{\text{raw}}=(p_1-p_0) \times (p_2-p_0)
$$

会得到 zero vector：

$$
n_{\text{raw}}=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

zero vector 不能作为 normal。

原因有两个：

1. zero vector 没有方向；
2. zero vector 不能 normalized。

因此计算 triangle normal 时必须检查：

$$
||n_{\text{raw}}|| \approx 0
$$

如果接近 0，就应该认为这是 degenerate triangle。

## 03.06.09 Concrete Example: Degenerate Triangle

给定：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
2 \\
2
\end{bmatrix}
$$

构造边：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
2 \\
2 \\
2
\end{bmatrix}
$$

可以看到：

$$
e_2=2e_1
$$

所以 e1 和 e2 共线。

因此：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

这个三角形没有有效的 surface normal。

## 03.06.10 Triangle Normal and Area

未归一化 normal 的长度和三角形面积有关。

因为：

$$
n_{\text{raw}}=e_1 \times e_2
$$

所以：

$$
||n_{\text{raw}}||=||e_1 \times e_2||
$$

这等于由 e1 和 e2 张成的平行四边形面积。

三角形面积为：

$$
A=\frac{1}{2}||n_{\text{raw}}||
$$

因此，n_raw 同时包含两类信息：

```text
direction:
triangle normal direction

magnitude:
twice the triangle area

## 03.06.15 Graphics Application: Flat Shading

如果一个三角形的所有 fragment 使用同一个 face normal，那么会得到 flat shading。

flat shading 的特点是：

```text
each triangle has one constant normal

## 03.06.16 Graphics Application: Smooth Shading Preview

smooth shading 通常使用 vertex normal。

vertex normal 可以通过相邻 face normals 平均得到。

例如，一个顶点相邻多个三角形，它们的 raw normals 为：

$$
n_1,\ n_2,\ n_3,\ \dots,\ n_k
$$

可以先累加：

$$
n_{\text{sum}}=\sum_{i=1}^{k} n_i
$$

然后 normalize：

$$
\hat{n}_{\text{vertex}}=\operatorname{normalize}(n_{\text{sum}})
$$

如果使用 raw normal 累加，那么面积更大的三角形会自动有更大权重。

因为：

$$
||n_i||=2A_i
$$

这就是 area-weighted vertex normal 的基本思想。

本节不展开实现，后续在 lighting and shading 中会继续讲。

## 03.06.17 Graphics Application: Back-face Culling

triangle normal 也用于 back-face culling。

设 view direction 是从 surface point 指向 camera 的单位方向：

$$
\hat{v}=\operatorname{normalize}(p_{\text{camera}}-p_{\text{surface}})
$$

如果：

$$
\hat{n} \cdot \hat{v}>0
$$

说明表面朝向相机。

如果：

$$
\hat{n} \cdot \hat{v}<0
$$

说明表面背对相机。

背对相机的三角形通常可以不绘制，这就是 back-face culling 的基本直觉。

注意：实际图形 API 中 front face 的判断还会受到 winding order、view space、clip space 和具体约定影响。这里先建立几何直觉。

## 03.06.20 Exercises

### Exercise 1

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

判断 normal 指向哪个方向。

---

### Exercise 2

如果把 Exercise 1 的顶点顺序改成：

```text
p0 → p2 → p1

请重新计算 normal，并说明方向如何变化。

# 03.07 Orientation, Winding Order, and Back-face Culling

前面我们已经知道，cross product 可以从三角形的两条边计算出一个垂直于三角形平面的 normal。

这一节继续讨论一个更接近渲染管线的问题：

给定同样的三个顶点，为什么不同的顶点顺序会让三角形“朝向”不同？

这涉及三个核心概念：

- orientation；
- winding order；
- back-face culling。

本节目标是理解：

- 三角形顶点顺序如何决定 normal direction；
- 什么是 front face 和 back face；
- 为什么背面三角形可以被剔除；
- normal direction 和 view direction 如何共同决定一个三角形是否朝向相机。

## 03.07.01 Graphics Motivation

在真实渲染中，一个 3D 模型通常由大量三角形组成。

例如：

- 一个 cube 可以由 12 个三角形组成；
- 一个角色模型可能有几千到几十万个三角形；
- 一个复杂场景可能有数百万个三角形。

但是从相机视角看，很多三角形其实背对相机。

如果一个三角形背对相机，那么在封闭模型中，它通常不会被看到。

因此渲染系统会尝试跳过这些三角形，减少后续计算量。

这个过程叫做：

```text
back-face culling

## 03.07.02 Orientation and Winding Order

orientation 可以理解为一个有顺序的几何对象的“朝向”。

对于三角形，三个顶点本身不仅仅是一组三个点，而是一个有顺序的点列。

例如：

```text
p0 → p1 → p2

和：
```text
p0 → p2 → p1

使用的是同样的三个点，但顶点顺序不同。
常见 winding order 包括：

counter-clockwise winding
clockwise winding

## 03.07.03 Normal Direction from Winding Order

给定三角形三个顶点：

$$
p_0,\ p_1,\ p_2
$$

构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

用 cross product 计算 raw normal：

$$
n_{\text{raw}}=e_1 \times e_2
$$

单位 normal 为：

$$
\hat{n}=\frac{n_{\text{raw}}}{\|n_{\text{raw}}\|}
$$

因此：

$$
\hat{n}=\frac{(p_1-p_0)\times(p_2-p_0)}{\|(p_1-p_0)\times(p_2-p_0)\|}
$$

这个 normal 的方向由顶点顺序 p0 → p1 → p2 决定。

如果交换 p1 和 p2：

$$
n'_{\text{raw}}=(p_2-p_0)\times(p_1-p_0)
$$

根据 cross product 的反交换性：

$$
(p_2-p_0)\times(p_1-p_0)=-((p_1-p_0)\times(p_2-p_0))
$$

所以：

$$
n'_{\text{raw}}=-n_{\text{raw}}
$$

也就是说：

```text
swapping two vertices flips the normal direction.

## 03.07.04 Core Proposition: Winding Order Flips the Normal

核心命题：

```text
The same triangle with opposite winding order has the opposite normal direction.

## 03.07.05 Front Face and Back Face

一个三角形有两个侧面：

```text
front face
back face
```

从几何直觉上说：如果 normal 指向相机一侧，这个面是 front-facing；如果 normal 背离相机一侧，这个面是 back-facing。

设三角形表面上的一点为 p_surface，相机位置为 p_camera。从表面指向相机的 view direction 定义为：

$$
v=\frac{p_{camera}−p_{surface}}{∥p_{camera}−p_{surface}∥}
$$

则：

n · v > 0 means front-facing.

n · v < 0 means back-facing.

## 03.07.06 Back-face Culling

back-face culling 是一种渲染优化。

它的思想是：

```text
If a triangle is facing away from the camera, skip it.
```

对于封闭物体，背向相机的三角形通常不会被看见。因此可以在进入更昂贵的 fragment shading 之前直接丢弃它们。所以一个简单的 culling rule 是：

```text
keep triangle if n · v > 0
cull triangle if n · v <= 0
```

实际图形 API 中，front face 的判断通常发生在投影之后的 screen space，并且依赖 API 的 winding order 约定。

例如：

```text
OpenGL 可以设置 CCW 或 CW 为 front face；
不同坐标系、投影矩阵、屏幕 y 轴方向可能影响最终屏幕空间 winding；
```

本课程当前先使用 3D 几何直觉理解 normal 和 view direction 的关系。,后续讲 view matrix、projection matrix 和 rasterization 时，再把 API 规则和屏幕空间 winding 连接起来。

## 03.07.07 Concrete Numerical Example

考虑一个位于 z = 0 平面上的三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

两条边为：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

计算 cross product：

$$
n=e_1\times e_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以这个 winding order 得到的 normal 指向 +z 方向。

如果相机在：

$$
p_{\text{camera}}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

三角形中心可以取：

$$
p_{\text{surface}}=\frac{p_0+p_1+p_2}{3}
$$

view direction 是从三角形表面指向相机：

$$
v=\frac{p_{\text{camera}}-p_{\text{surface}}}{\|p_{\text{camera}}-p_{\text{surface}}\|}
$$

由于相机在三角形的 +z 侧，v 大致指向 +z。

因此：

$$
n\cdot v>0
$$

这个三角形朝向相机。

如果交换 p1 和 p2，那么：

$$
n'=(p_2-p_0)\times(p_1-p_0)=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

此时：

$$
n'\cdot v<0
$$

所以同样的三个点，换一个 winding order 后，就变成背对相机。

## 03.07.09 Graphics Application: Why Culling Matters

back-face culling 是渲染管线中的重要优化。一个封闭 mesh 的许多三角形在当前相机视角下是背对相机的。

如果不剔除它们，GPU 可能仍然需要对这些三角形执行后续处理，例如：

- vertex processing；
- clipping；
- rasterization；
- fragment shading。

其中 fragment shading 往往很昂贵。

因此，如果能提前判断某些三角形不可见，就可以节省大量计算。

直觉上：

```text
front-facing triangles should be rendered.
back-facing triangles can often be skipped.
```

但是 back-face culling 并不总是适用。

例如：
```text
纸张；
树叶；
布料；
玻璃；
双面材质；
某些特效面片；
```
这些对象可能需要双面渲染。

所以 back-face culling 是一种常用优化，但不是所有材质都应该开启。

## 03.07.11 Exercises

### Exercise 1.

给定：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请计算：

$$
(p_1-p_0)\times(p_2-p_0)
$$

并判断 normal 指向哪个方向。

---

### Exercise 2.

仍然使用 Exercise 1 的三个点，但交换 p1 和 p2。

请计算：

$$
(p_2-p_0)\times(p_1-p_0)
$$

并说明它和 Exercise 1 的结果有什么关系。

---

### Exercise 3.

设相机位置为：

$$
p_{\text{camera}}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

三角形位于 z = 0 平面，normal 为：

$$
\hat{n}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请判断这个三角形是否朝向相机。

---

### Exercise 4.

设相机位置仍然为：

$$
p_{\text{camera}}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

但三角形 normal 为：

$$
\hat{n}=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

请判断这个三角形是否朝向相机。

---

### Exercise 5.

请解释为什么 back-face culling 可以提高渲染效率。

要求从下面几个角度说明：

- 三角形数量；
- 相机可见性；
- fragment shading 成本；
- 封闭 mesh 的背面通常不可见。

---

### Exercise 6.

请用 NumPy 实现一个函数：

```text
is_front_facing(p0, p1, p2, camera_position)

# 03.08 NumPy Implementation

前面几节我们已经学习了 cross product 在图形学中的核心用途：

- 计算垂直方向；
- 计算三角形 raw normal；
- 计算单位 triangle normal；
- 计算三角形面积；
- 判断 degenerate triangle；
- 理解 winding order 如何影响 normal direction；
- 判断三角形是否 front-facing 或 back-facing。

这一节的目标是把 Chapter 03 的数学内容整理成一组可以放入 `src/vector_ops.py` 的 NumPy 函数。

本节对应文件建议为：

- `src/vector_ops.py`
- `notebooks/03_cross_product_orientation_surface_normals.ipynb`

本节重点不是引入新数学概念，而是把已经学过的内容工程化。

## 03.08.03 Vector Validation Helper

为了避免每个函数重复写 shape 检查，可以先定义一个内部辅助函数：

`_as_vec3(x, name="vector")`

它负责三件事：

1. 把输入转成 `np.asarray(..., dtype=float)`；
2. 检查 shape 是否为 `(3,)`；
3. 返回标准化后的 NumPy array。

注意这里的“标准化”不是 normalize 成单位向量，而是把输入格式标准化为 3D NumPy 向量。

这个函数以下划线开头，表示它主要供模块内部使用，不一定作为公开 API。

# 03.09 Graphics Application: Surface Normals and Camera Basis

本章前面一直在学习 cross product 的数学性质：

- cross product 输出一个垂直于两个输入向量的方向；
- cross product 的长度表示面积；
- cross product 的方向由 right-hand rule 决定；
- triangle normal 由 winding order 决定；
- normal direction 可以用于判断 front-facing / back-facing。

这一节把这些内容连接到两个图形学应用：

1. surface normals；
2. camera basis。

这两个对象都会在后续课程反复出现：

- 光照需要 normal；
- back-face culling 需要 normal；
- smooth shading 需要 vertex normal；
- normal mapping 需要 tangent space；
- view matrix 需要 camera basis；
- ray generation 需要 camera basis；
- billboard 和 object orientation 也需要 local frame。

本节的核心问题是：

如何用 cross product 从已有方向构造新的几何方向？

## 03.09.01 Surface Normal as a Geometric Direction

surface normal 是垂直于表面的方向。

对于一个三角形：

$$
p_0,\ p_1,\ p_2
$$

先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

然后计算 raw normal：

$$
n_{\text{raw}}=e_1\times e_2
$$

单位 normal 为：

$$
\hat{n}=\frac{n_{\text{raw}}}{\|n_{\text{raw}}\|}
$$

surface normal 的作用不是描述位置，而是描述方向。

因此它是 vector，不是 point。

在 homogeneous coordinates 中，normal 不是普通 point：

$$
n=
\begin{bmatrix}
n_x \\
n_y \\
n_z \\
0
\end{bmatrix}
$$

最后一维是 0，表示它不受 translation 影响。

这是因为平移物体不会改变表面的朝向。

## 03.09.02 Face Normal and Flat Shading

face normal 是每个 triangle 一个 normal。

对于一个三角形：

$$
\hat{n}_{\text{face}}=
\frac{(p_1-p_0)\times(p_2-p_0)}
{\|(p_1-p_0)\times(p_2-p_0)\|}
$$

如果一个 mesh 使用 face normal 进行光照，那么同一个三角形内部所有位置使用同一个 normal。

这种效果叫做：

flat shading

flat shading 的特点是：

- 每个三角形看起来是平的；
- 三角形之间的边界比较明显；
- 适合 low-poly 风格；
- 适合明确表现几何面片结构。

但是对于球体、角色、曲面等对象，flat shading 可能会显得棱角明显。

这时通常需要 vertex normal。

## 03.09.03 Vertex Normal and Smooth Shading

vertex normal 是每个 vertex 一个 normal。

它通常由这个顶点相邻的多个 face normal 平均得到。

假设某个顶点周围有多个三角形，它们的 face normal 分别为：

$$
\hat{n}_1,\hat{n}_2,\hat{n}_3,\dots,\hat{n}_k
$$

一种简单的 vertex normal 计算方式是：

$$
n_{\text{vertex, raw}}=\sum_{i=1}^{k}\hat{n}_i
$$

然后归一化：

$$
\hat{n}_{\text{vertex}}=
\frac{n_{\text{vertex, raw}}}{\|n_{\text{vertex, raw}}\|}
$$

更常见的做法是 area-weighted average。

因为 raw face normal 的长度等于三角形面积的两倍：

$$
\|n_{\text{raw}}\|=2A
$$

所以可以直接累加 raw normal：

$$
n_{\text{vertex, raw}}=\sum_{i=1}^{k} n_{\text{face raw}, i}
$$

然后再归一化：

$$
\hat{n}_{\text{vertex}}=
\frac{n_{\text{vertex, raw}}}{\|n_{\text{vertex, raw}}\|}
$$

这样面积更大的三角形对 vertex normal 的影响更大。

smooth shading 的核心思想是：

不是直接用真实三角形平面的 normal，而是用插值得到的 vertex normal 让表面看起来更光滑。

## 03.09.04 Numerical Example: Face Normal

考虑三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

两条边为：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

cross product 为：

$$
e_1\times e_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以该三角形的单位 normal 是：

$$
\hat{n}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

这说明在当前 winding order 下，这个三角形朝向 +z 方向。

## 03.09.05 Surface Normal and Lighting

surface normal 最重要的应用之一是光照。

Lambert diffuse lighting 的核心公式是：

$$
I=\max(0,\hat{n}\cdot \hat{l})
$$

其中：

- $\hat{n}$ 是单位 surface normal；
- $\hat{l}$ 是单位 light direction；
- 本课程约定 light direction 是 from surface point to light source；
- I 是 diffuse intensity。

如果：

$$
\hat{n}\cdot \hat{l}=1
$$

说明表面正对光源，亮度最大。

如果：

$$
\hat{n}\cdot \hat{l}=0
$$

说明光线沿着表面切过去，diffuse intensity 为 0。

如果：

$$
\hat{n}\cdot \hat{l}<0
$$

说明光源在表面背面，使用 max 截断为 0。

所以 normal 不只是几何方向，它直接参与 shading。

## 03.09.06 Camera Basis as an Orthonormal Frame

cross product 的另一个重要应用是构造 camera basis。

相机不是只有一个 position。

一个 3D camera 至少需要：

- position；
- forward direction；
- right direction；
- up direction。

这三个方向构成 camera frame。

在本项目的右手坐标系约定下，我们采用下面的 camera basis：

- r：camera right；
- u：camera up；
- b：camera backward。

注意这里使用 backward，而不是 forward。

原因是很多右手图形学系统中，camera 在 view space 中看向 -z 方向。

也就是说：

$$
\text{camera forward}=-b
$$

camera basis 满足：

$$
r\times u=b
$$

这说明：

$$
r,\ u,\ b
$$

构成一个 right-handed orthonormal frame。

其中：

- r 是单位 right vector；
- u 是单位 up vector；
- b 是单位 backward vector。

## 03.09.07 Constructing Camera Basis from Eye, Target, and World Up

通常我们用三个输入定义相机：

- eye：相机位置；
- target：相机看向的目标点；
- world_up：世界空间中的参考上方向。

首先计算 camera backward direction：

$$
b=\frac{eye-target}{\|eye-target\|}
$$

这里 b 是从 target 指向 eye 的方向，也就是 camera backward。

camera forward direction 是：

$$
f=-b
$$

然后用 world_up 和 b 计算 right direction：

$$
r=\frac{world\_up\times b}{\|world\_up\times b\|}
$$

最后用 b 和 r 重新计算 camera up：

$$
u=b\times r
$$

这样得到的：

$$
r,\ u,\ b
$$

满足：

$$
r\times u=b
$$

并且通常是 orthonormal 的。

注意：

不要直接把 world_up 当作 camera up。

world_up 只是一个参考方向。

真正的 camera up 必须同时垂直于 camera right 和 camera backward。

## 03.09.08 Numerical Example: Camera Basis

设相机位置为：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

看向目标点：

$$
target=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

世界上方向为：

$$
world\_up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

首先计算 backward：

$$
b=\frac{eye-target}{\|eye-target\|}
$$

因此：

$$
b=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

然后：

$$
r=\frac{world\_up\times b}{\|world\_up\times b\|}
$$

也就是：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

最后：

$$
u=b\times r
$$

得到：

$$
u=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

所以相机 basis 为：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix},
\quad
u=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix},
\quad
b=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

这个例子中，相机位于 +z 方向，看向原点，因此 camera forward 是：

$$
f=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

## 03.09.13 Exercises

### Exercise 1.

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请计算：

$$
n_{\text{raw}}=(p_1-p_0)\times(p_2-p_0)
$$

并说明：

1. raw normal 的方向；
2. raw normal 的长度；
3. 三角形面积。

---

### Exercise 2.

为什么 face normal 可以用于 flat shading？

请从下面角度说明：

- 一个 triangle 内部使用同一个 normal；
- 相邻 triangle 的 normal 可能不同；
- 为什么会产生棱角感。

---

### Exercise 3.

为什么 vertex normal 可以用于 smooth shading？

请从下面角度说明：

- 一个 vertex normal 可以由多个 face normal 平均得到；
- rasterization 时可以插值 vertex normal；
- 插值后的 normal 会让曲面看起来更连续。

---

### Exercise 4.

设：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

$$
target=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
world\_up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请计算 camera basis：

$$
r,\ u,\ b
$$

并验证：

$$
r\cdot u=0
$$

$$
u\cdot b=0
$$

$$
b\cdot r=0
$$

---

### Exercise 5.

为什么不能总是直接使用 world_up 作为 camera up？

请说明当 world_up 不垂直于 camera direction 时会发生什么问题。

---

### Exercise 6.

请用 NumPy 实现：

`camera_basis(eye, target, world_up)`

要求：

1. 返回 camera right r；
2. 返回 camera up u；
3. 返回 camera backward b；
4. 检查 world_up 是否和 camera direction 平行；
5. 验证 r, u, b 是否互相垂直。

---

### Exercise 7.

设 camera basis 为：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
u=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
b=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

相机位置为：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

世界空间点为：

$$
p=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

请计算 p 在 camera basis 下的坐标：

$$
p_{\text{view}}=
\begin{bmatrix}
r\cdot(p-eye) \\
u\cdot(p-eye) \\
b\cdot(p-eye)
\end{bmatrix}
$$

并解释为什么结果的 z 分量是负数或正数，取决于你使用 forward 还是 backward 作为 camera z axis。

## 03.09.13 Exercises

### Exercise 1.

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请计算：

$$
n_{\text{raw}}=(p_1-p_0)\times(p_2-p_0)
$$

并说明：

1. raw normal 的方向；
2. raw normal 的长度；
3. 三角形面积。

---

### Exercise 2.

为什么 face normal 可以用于 flat shading？

请从下面角度说明：

- 一个 triangle 内部使用同一个 normal；
- 相邻 triangle 的 normal 可能不同；
- 为什么会产生棱角感。

---

### Exercise 3.

为什么 vertex normal 可以用于 smooth shading？

请从下面角度说明：

- 一个 vertex normal 可以由多个 face normal 平均得到；
- rasterization 时可以插值 vertex normal；
- 插值后的 normal 会让曲面看起来更连续。

---

### Exercise 4.

设：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

$$
target=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
world\_up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请计算 camera basis：

$$
r,\ u,\ b
$$

并验证：

$$
r\cdot u=0
$$

$$
u\cdot b=0
$$

$$
b\cdot r=0
$$

---

### Exercise 5.

为什么不能总是直接使用 world_up 作为 camera up？

请说明当 world_up 不垂直于 camera direction 时会发生什么问题。

---

### Exercise 6.

请用 NumPy 实现：

`camera_basis(eye, target, world_up)`

要求：

1. 返回 camera right r；
2. 返回 camera up u；
3. 返回 camera backward b；
4. 检查 world_up 是否和 camera direction 平行；
5. 验证 r, u, b 是否互相垂直。

---

### Exercise 7.

设 camera basis 为：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
u=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
b=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

相机位置为：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

世界空间点为：

$$
p=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

请计算 p 在 camera basis 下的坐标：

$$
p_{\text{view}}=
\begin{bmatrix}
r\cdot(p-eye) \\
u\cdot(p-eye) \\
b\cdot(p-eye)
\end{bmatrix}
$$

并解释为什么结果的 z 分量是负数或正数，取决于你使用 forward 还是 backward 作为 camera z axis。

## 03.09.14 Summary

本节将 cross product 连接到了两个重要图形学应用：

1. surface normals；
2. camera basis。

对于 surface normal：

$$
n=(p_1-p_0)\times(p_2-p_0)
$$

cross product 用两条三角形边构造垂直于表面的方向。

对于 camera basis：

$$
b=\frac{eye-target}{\|eye-target\|}
$$

$$
r=\frac{world\_up\times b}{\|world\_up\times b\|}
$$

$$
u=b\times r
$$

cross product 用已有方向构造相机的 right 和 up。

本节最重要的结论是：

cross product 不只是一个代数公式，它是图形学中构造 frame、normal、orientation 的基本工具。

下一节将进入：

`03.10 Common Mistakes`

我们会集中整理 Chapter 03 中关于 cross product、normal、winding order、camera basis 最容易犯错的地方。

# 03.11 Exercises

本节练习用于检查 Chapter 03 的核心内容：

- cross product；
- right-hand rule；
- cross product magnitude；
- triangle normal；
- raw normal vs unit normal；
- winding order；
- front face vs back face；
- back-face culling；
- surface normal；
- camera basis；
- degenerate triangle；
- NumPy implementation。

请先独立完成，不要直接看答案。

下一节 `03.11 Exercises Answers` 会集中给出推导过程、数值结果和 NumPy 验证。

## 03.11.01 Exercise 1: Basic Cross Product Direction

给定右手坐标系中的三个单位坐标轴：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
z=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请计算下面结果，并说明方向：

$$
x\times y
$$

$$
y\times x
$$

$$
y\times z
$$

$$
z\times y
$$

$$
z\times x
$$

$$
x\times z
$$

要求：

1. 写出每个 cross product 的结果；
2. 标明结果是 +x、-x、+y、-y、+z 还是 -z；
3. 说明为什么 cross product 的顺序不能随意交换。

## 03.11.02 Exercise 2: Cross Product Algebraic Computation

给定：

$$
u=
\begin{bmatrix}
2 \\
1 \\
3
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
1 \\
4 \\
-2
\end{bmatrix}
$$

请根据 cross product 的代数定义计算：

$$
u\times v
$$

其中：

$$
u\times v=
\begin{bmatrix}
u_yv_z-u_zv_y \\
u_zv_x-u_xv_z \\
u_xv_y-u_yv_x
\end{bmatrix}
$$

要求：

1. 手算每一个分量；
2. 写出最终向量；
3. 再计算 `v × u`；
4. 验证：

$$
u\times v=-(v\times u)
$$

## 03.11.03 Exercise 3: Orthogonality of Cross Product

延续 Exercise 2 的向量：

$$
u=
\begin{bmatrix}
2 \\
1 \\
3
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
1 \\
4 \\
-2
\end{bmatrix}
$$

设：

$$
w=u\times v
$$

请验证：

$$
w\cdot u=0
$$

$$
w\cdot v=0
$$

要求：

1. 使用 Exercise 2 中得到的 w；
2. 分别计算 w 与 u 的 dot product；
3. 分别计算 w 与 v 的 dot product；
4. 解释为什么这说明 w 垂直于 u 和 v。

## 03.11.04 Exercise 4: Cross Product Magnitude and Area

给定：

$$
u=
\begin{bmatrix}
3 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
4 \\
0
\end{bmatrix}
$$

请计算：

$$
u\times v
$$

以及：

$$
\|u\times v\|
$$

要求：

1. 写出 cross product 的结果；
2. 计算 cross product 的长度；
3. 说明 u 和 v 张成的平行四边形面积是多少；
4. 说明由 u 和 v 构成的三角形面积是多少；
5. 验证结果是否符合：

$$
\|u\times v\|=\|u\|\|v\|\sin(\theta)
$$

## 03.11.05 Exercise 5: Triangle Raw Normal and Unit Normal

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请计算两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

然后计算 raw normal：

$$
n_{\text{raw}}=e_1\times e_2
$$

再计算 unit normal：

$$
\hat{n}=\frac{n_{\text{raw}}}{\|n_{\text{raw}}\|}
$$

要求：

1. 写出 e1 和 e2；
2. 写出 raw normal；
3. 写出 raw normal 的长度；
4. 写出 unit normal；
5. 说明这个三角形朝向 +z 还是 -z。

## 03.11.06 Exercise 6: Triangle Area from Raw Normal

延续 Exercise 5 的三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请使用 raw normal 计算三角形面积。

已知：

$$
A=\frac{1}{2}\|n_{\text{raw}}\|
$$

要求：

1. 使用 Exercise 5 中的 raw normal；
2. 计算三角形面积；
3. 用底乘高除以 2 的方式验证结果；
4. 说明为什么 raw normal 的长度等于三角形面积的两倍。

## 03.11.07 Exercise 7: Winding Order and Normal Direction

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

先计算：

$$
n=(p_1-p_0)\times(p_2-p_0)
$$

再交换 p1 和 p2，计算：

$$
n'=(p_2-p_0)\times(p_1-p_0)
$$

要求：

1. 写出 n；
2. 写出 n'；
3. 验证：

$$
n'=-n
$$

4. 解释为什么同样的三个点，换一个顶点顺序会改变 normal direction。

## 03.11.08 Exercise 8: Front-facing and Back-facing

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

相机位置为：

$$
p_{\text{camera}}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

三角形表面点用中心点近似：

$$
p_{\text{surface}}=\frac{p_0+p_1+p_2}{3}
$$

view direction 定义为：

$$
v=\frac{p_{\text{camera}}-p_{\text{surface}}}{\|p_{\text{camera}}-p_{\text{surface}}\|}
$$

要求：

1. 计算三角形 normal；
2. 计算三角形中心点；
3. 写出 view direction 的方向直觉；
4. 判断：

$$
\hat{n}\cdot v>0
$$

还是：

$$
\hat{n}\cdot v<0
$$

5. 判断该三角形是 front-facing 还是 back-facing。

## 03.11.09 Exercise 9: Back-face Culling Conceptual Question

请用自己的话解释 back-face culling。

要求回答下面问题：

1. back-face culling 想要跳过哪些三角形？
2. 为什么封闭不透明 mesh 通常适合开启 back-face culling？
3. 为什么纸张、树叶、布料、玻璃或特效面片可能不适合只渲染单面？
4. front face 和 visible face 是不是完全相同的概念？
5. back-face culling 是数学真理，还是一种渲染优化策略？

## 03.11.10 Exercise 10: Degenerate Triangle

给定三个点：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

请判断这个三角形是否退化。

要求：

1. 计算：

$$
e_1=p_1-p_0
$$

2. 计算：

$$
e_2=p_2-p_0
$$

3. 计算：

$$
n_{\text{raw}}=e_1\times e_2
$$

4. 判断：

$$
\|n_{\text{raw}}\|\approx 0
$$

是否成立；

5. 解释为什么 degenerate triangle 不能计算有效 unit normal。

## 03.11.11 Exercise 11: Surface Normal and Lambert Lighting

设表面单位法线为：

$$
\hat{n}=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

分别考虑三个 light direction：

$$
\hat{l}_1=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

$$
\hat{l}_2=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
\hat{l}_3=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

Lambert diffuse intensity 为：

$$
I=\max(0,\hat{n}\cdot \hat{l})
$$

要求：

1. 分别计算三个 dot product；
2. 分别计算三个 Lambert intensity；
3. 解释哪种情况下表面最亮；
4. 解释哪种情况下表面完全不受光；
5. 明确本题中的 light direction 约定是 from surface point to light source。

## 03.11.12 Exercise 12: Camera Basis Construction

给定：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

$$
target=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
world\_up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

按照本课程右手相机约定：

$$
b=\frac{eye-target}{\|eye-target\|}
$$

$$
r=\frac{world\_up\times b}{\|world\_up\times b\|}
$$

$$
u=b\times r
$$

要求：

1. 计算 camera backward b；
2. 计算 camera right r；
3. 计算 camera up u；
4. 写出 camera forward f；
5. 验证：

$$
r\cdot u=0
$$

$$
u\cdot b=0
$$

$$
b\cdot r=0
$$

6. 验证：

$$
r\times u=b
$$

## 03.11.13 Exercise 13: Camera Basis Degeneracy

考虑下面相机输入：

$$
eye=
\begin{bmatrix}
0 \\
5 \\
0
\end{bmatrix}
$$

$$
target=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
world\_up=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

要求：

1. 计算：

$$
b=\frac{eye-target}{\|eye-target\|}
$$

2. 判断 world_up 是否与 b 平行；
3. 计算：

$$
world\_up\times b
$$

4. 判断是否能够构造 camera right r；
5. 解释为什么这种情况下需要选择一个 fallback up direction。

## 03.11.14 Exercise 14: World Point to Camera Coordinates

给定 camera basis：

$$
r=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
u=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
b=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

相机位置为：

$$
eye=
\begin{bmatrix}
0 \\
0 \\
5
\end{bmatrix}
$$

世界空间点为：

$$
p=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

使用公式：

$$
p_{\text{view}}=
\begin{bmatrix}
r\cdot(p-eye) \\
u\cdot(p-eye) \\
b\cdot(p-eye)
\end{bmatrix}
$$

要求：

1. 计算 p - eye；
2. 计算 x_view；
3. 计算 y_view；
4. 计算 z_view；
5. 解释这个点为什么位于相机前方；
6. 说明为什么在这个约定下，相机前方对应负 z。